# Thread Pool Executor and Process Pool

This notebook explains how to use `ThreadPoolExecutor` and `ProcessPoolExecutor` from Python's `concurrent.futures` module. It shows when each tool is useful and gives small examples you can run locally.

## Learning Goals

- Use `ThreadPoolExecutor` for I/O-bound work
- Use `ProcessPoolExecutor` for CPU-bound work
- Collect results with `submit()` and `as_completed()`
- Compare pool-based concurrency with serial execution
- Keep multiprocessing code Windows-safe

In [1]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from time import perf_counter, sleep

## What Is a Pool Executor?

A pool executor manages a group of worker threads or worker processes for you. Instead of creating and joining workers manually, you submit tasks to the pool and collect results when they finish.

Use a thread pool when tasks mostly wait on I/O. Use a process pool when tasks spend most of their time using the CPU.

## Example 1: Thread Pool for I/O-Bound Work

This example simulates slow I/O with `sleep()`. Thread pools are a good fit because the workers spend time waiting instead of using the CPU heavily.

In [2]:
def io_task(name, delay):
    print(f'{name} started')
    sleep(delay)
    print(f'{name} finished')
    return f'{name} result'

jobs = [(f'Job-{index}', 1) for index in range(1, 5)]

start = perf_counter()
serial_results = [io_task(name, delay) for name, delay in jobs]
serial_time = perf_counter() - start

print(f'Serial execution took {serial_time:.2f} seconds')
print('Serial results:', serial_results)

Job-1 started
Job-1 finished
Job-2 started
Job-2 finished
Job-3 started
Job-3 finished
Job-4 started
Job-4 finished
Serial execution took 4.00 seconds
Serial results: ['Job-1 result', 'Job-2 result', 'Job-3 result', 'Job-4 result']


In [3]:
start = perf_counter()
thread_results = []

with ThreadPoolExecutor(max_workers=4) as executor:
    future_to_job = {executor.submit(io_task, name, delay): name for name, delay in jobs}
    for future in as_completed(future_to_job):
        thread_results.append(future.result())

thread_time = perf_counter() - start

print(f'Thread pool execution took {thread_time:.2f} seconds')
print('Thread pool results:', thread_results)

Job-1 started
Job-2 started
Job-3 started
Job-4 started
Job-1 finished
Job-2 finished
Job-3 finished
Job-4 finished
Thread pool execution took 1.00 seconds
Thread pool results: ['Job-1 result', 'Job-2 result', 'Job-3 result', 'Job-4 result']


## Example 2: Process Pool for CPU-Bound Work

This example uses a simple CPU-heavy calculation. Process pools can run tasks in separate interpreter processes, which helps with true parallel execution on multiple CPU cores.

On Windows, keep the process-pool code inside a main guard. The example below follows that rule.

In [4]:
def cpu_task(number):
    total = 0
    for value in range(number):
        total += value * value
    return total

numbers = [2000000, 2200000, 2400000, 2600000]

if __name__ == '__main__':
    start = perf_counter()
    serial_cpu_results = [cpu_task(number) for number in numbers]
    serial_cpu_time = perf_counter() - start

    print(f'Serial CPU work took {serial_cpu_time:.2f} seconds')
    print('Serial CPU results:', serial_cpu_results)

Serial CPU work took 0.47 seconds
Serial CPU results: [2666664666667000000, 3549330913333700000, 4607997120000400000, 5858663286667100000]


In [5]:
if __name__ == '__main__':
    start = perf_counter()
    process_results = []

    with ProcessPoolExecutor(max_workers=4) as executor:
        future_to_number = {executor.submit(cpu_task, number): number for number in numbers}
        for future in as_completed(future_to_number):
            process_results.append(future.result())

    process_time = perf_counter() - start

    print(f'Process pool execution took {process_time:.2f} seconds')
    print('Process pool results:', process_results)

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

## Key Differences

- `ThreadPoolExecutor` is lightweight and works well for waiting-heavy tasks
- `ProcessPoolExecutor` is heavier but better for CPU-heavy work
- Both executors let you submit tasks and gather results without managing workers manually
- Functions sent to a process pool must be picklable and defined at the top level

## Practical Takeaways

- Use a thread pool for downloads, API calls, file waits, and other I/O-heavy jobs.
- Use a process pool for number crunching, image processing, and other CPU-heavy jobs.
- Keep pool task functions simple, top-level, and easy to test.
- Always respect the Windows multiprocessing main guard in scripts and notebooks.